# Motor Overheating Prediction
**Pipeline:** XGBoost + inyeccion de fallos + split por secuencia (80/20).

Ejecuta las celdas de arriba abajo la primera vez. Despues puedes re-ejecutar cada motor por separado sin repetir la carga de datos.

## 1. Imports

In [40]:
import os, sys, warnings, itertools
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from scipy.ndimage import binary_closing

warnings.filterwarnings('ignore')
sys.path.insert(1, './kaggle_data_challenge')
from utility import read_all_test_data_from_path

print('Imports OK')
print(f'XGBoost {xgb.__version__}')

Imports OK
XGBoost 3.2.0


## 2. Funciones auxiliares

In [41]:
def inject_failure(temperature_values, label_values):
    """Porta el algoritmo MATLAB de kaggle.md: subida en 1/3 de la region de fallo, bajada en 2/3."""
    failure_mask = (label_values == 1)
    if failure_mask.sum() == 0:
        return temperature_values.copy()
    tmp_temp   = temperature_values[failure_mask]
    n_seq      = len(tmp_temp)
    n_rise     = n_seq // 3
    n_fall     = n_seq - n_rise
    temp_start = tmp_temp[0]
    temp_end   = tmp_temp[-1]
    temp_high  = max(temp_start, temp_end) + np.random.randint(2, 11)
    rise       = np.linspace(temp_start, temp_high, n_rise + 1)[:-1]
    fall       = np.linspace(temp_high,  temp_end,  n_fall)
    injected   = np.concatenate([rise, fall])
    result     = temperature_values.copy()
    result[failure_mask] = injected
    return result


def pre_processing(df):
    if len(df) == 0:
        return
    df['temperature'] = df['temperature'].where(df['temperature'] <= 100, np.nan)
    df['temperature'] = df['temperature'].where(df['temperature'] >= 0,   np.nan)
    df['temperature'] = df['temperature'].ffill()
    df['voltage']     = df['voltage'].where(df['voltage'] >= 6000, np.nan)
    df['voltage']     = df['voltage'].where(df['voltage'] <= 9000, np.nan)
    df['voltage']     = df['voltage'].ffill()
    df['position']    = df['position'].where(df['position'] >= 0,    np.nan)
    df['position']    = df['position'].where(df['position'] <= 1000, np.nan)
    df['position']    = df['position'].ffill()
    df['temperature'] -= df['temperature'].iloc[0]
    df['voltage']     -= df['voltage'].iloc[0]
    df['position']    -= df['position'].iloc[0]
    df['temperature_diff'] = df['temperature'].diff(20)
    df['voltage_diff']     = df['voltage'].diff(20)
    df['position_diff']    = df['position'].diff(20)
    for w, s in [(5, '5'), (20, '20')]:
        df[f'temperature_roll_mean_{s}'] = df['temperature'].rolling(w, min_periods=1).mean()
        df[f'temperature_roll_max_{s}']  = df['temperature'].rolling(w, min_periods=1).max()
        df[f'temperature_roll_std_{s}']  = df['temperature'].rolling(w, min_periods=1).std().fillna(0)
    n = len(df)
    df['time_pct'] = np.linspace(0.0, 1.0, n) if n > 1 else np.zeros(n)
    df.fillna(0, inplace=True)


print('Funciones definidas')

Funciones definidas


## 3. Configuracion
Verifica que las rutas existen antes de continuar.

In [42]:
rolling_feats = [
    'temperature_roll_mean_5', 'temperature_roll_max_5', 'temperature_roll_std_5',
    'temperature_roll_mean_20', 'temperature_roll_max_20', 'temperature_roll_std_20',
]
MOTOR_FEATURES = {
    1: ['temperature', 'position', 'voltage', 'temperature_diff', 'time_pct'] + rolling_feats,
    2: ['temperature', 'position', 'voltage', 'time_pct'] + rolling_feats,
    3: ['position', 'temperature_diff', 'voltage', 'temperature', 'time_pct'] + rolling_feats,
    4: ['position', 'temperature', 'voltage', 'time_pct'] + rolling_feats,
    5: ['position', 'voltage', 'temperature', 'temperature_diff', 'time_pct'] + rolling_feats,
    6: ['position', 'temperature', 'temperature_diff', 'position_diff', 'time_pct'] + rolling_feats,
}

# Motores donde clasificacion supervisada no transfiere entre secuencias -> anomaly detection
BAD_MOTORS = [3, 4, 6]

base_dir        = './kaggle_data_challenge/kaggle_data_challenge'
train_path      = base_dir + '/training_data/'
test_path       = base_dir + '/testing_data/'
additional_base = './kaggle_data_challenge/additional_data'

models           = {}
val_f1_per_motor = {}

print('Config OK')
print(f'  BAD_MOTORS (anomaly detection): {BAD_MOTORS}')
print(f'  train_path existe : {os.path.exists(train_path)}')
print(f'  test_path  existe : {os.path.exists(test_path)}')

Config OK
  BAD_MOTORS (anomaly detection): [3, 4, 6]
  train_path existe : True
  test_path  existe : True


## 4. Carga de datos
Comprueba shape y distribucion de etiquetas antes de continuar.

In [43]:
import shutil

print('=== Datos de entrenamiento ===')
train_df = read_all_test_data_from_path(train_path, pre_processing, is_plot=False)
print(f'Shape base: {train_df.shape}')

groups = [
    'additional_data_20240524_group_6',
    'additional_training_data_group_1',
    'additional_training_data_group_7',
]
additional_dfs = []
for group in groups:
    group_path = os.path.join(additional_base, group)
    if not os.path.exists(group_path):
        print(f'  [{group}] no encontrado, omitiendo')
        continue
    xlsx      = os.path.join(group_path, 'Test conditions.xlsx')
    xlsx_copy = os.path.join(group_path, 'Test conditions copy.xlsx')
    if not os.path.exists(xlsx) and os.path.exists(xlsx_copy):
        try:
            shutil.copy(xlsx_copy, xlsx)
        except Exception as e:
            print(f'  copia fallida: {e}')
    tmp = os.path.join(additional_base, f'{group}_tmp')
    shutil.rmtree(tmp, ignore_errors=True)
    os.makedirs(tmp, exist_ok=True)
    if os.path.exists(xlsx):
        shutil.copy(xlsx, os.path.join(tmp, 'Test conditions.xlsx'))
    subdirs = [d for d in os.listdir(group_path) if os.path.isdir(os.path.join(group_path, d))]
    ok = 0
    for sd in subdirs:
        sp   = os.path.join(group_path, sd)
        csvs = [f for f in os.listdir(sp) if f.endswith('.csv')]
        valid = True
        for c in csvs:
            try:
                with open(os.path.join(sp, c)) as fh:
                    lines = [l.strip() for l in fh if l.strip()]
                if len(lines) <= 1:
                    valid = False; break
            except Exception:
                valid = False; break
        if valid and csvs:
            shutil.copytree(sp, os.path.join(tmp, sd))
            ok += 1
    print(f'  [{group}] {ok}/{len(subdirs)} subdirs validos')
    if ok > 0:
        try:
            gdf = read_all_test_data_from_path(tmp + '/', pre_processing, is_plot=False)
            additional_dfs.append(gdf)
        except Exception as e:
            print(f'  carga fallida: {e}')
    shutil.rmtree(tmp, ignore_errors=True)

if additional_dfs:
    train_df = pd.concat([train_df] + additional_dfs, ignore_index=True)
    print(f'Shape combinado: {train_df.shape}')

print()
print('=== Datos de test ===')
test_df = read_all_test_data_from_path(test_path, pre_processing, is_plot=False)
print(f'Shape: {test_df.shape}')

print()
print('=== Distribucion de etiquetas por motor ===')
print(f'{"Motor":<8} {"Normal":>8} {"Fallo":>8} {"% fallo":>9}')
print('-' * 38)
for mid in range(1, 7):
    col = f'data_motor_{mid}_label'
    if col in train_df.columns:
        c0  = (train_df[col] == 0).sum()
        c1  = (train_df[col] == 1).sum()
        pct = 100 * c1 / (c0 + c1) if (c0 + c1) > 0 else 0
        print(f'{mid:<8} {c0:>8} {c1:>8} {pct:>8.1f}%')

=== Datos de entrenamiento ===
Shape base: (39309, 86)
  [additional_data_20240524_group_6] no encontrado, omitiendo
  [additional_training_data_group_1] no encontrado, omitiendo
  [additional_training_data_group_7] no encontrado, omitiendo

=== Datos de test ===
Shape: (14157, 86)

=== Distribucion de etiquetas por motor ===
Motor      Normal    Fallo   % fallo
--------------------------------------
1           37960     1349      3.4%
2           32577     6732     17.1%
3           39182      127      0.3%
4           32570     6739     17.1%
5           39125      184      0.5%
6           37377     1932      4.9%


## 5. Funcion train_motor
Muestra: split de secuencias -> inyeccion -> balance -> progreso del grid search -> F1 de validacion.

In [44]:
def train_motor(motor_id):
    sep  = '=' * 55
    mode = 'anomalia' if motor_id in BAD_MOTORS else 'supervisado'
    print()
    print(sep)
    print(f'  MOTOR {motor_id}  ({mode})')
    print(sep)

    features  = [f'data_motor_{motor_id}_{f}' for f in MOTOR_FEATURES[motor_id]]
    label_col = f'data_motor_{motor_id}_label'

    missing = [f for f in features if f not in train_df.columns]
    if missing:
        print(f'SKIP -- columnas faltantes: {missing}'); return
    if label_col not in train_df.columns:
        print('SKIP -- sin columna de etiqueta'); return

    motor_df  = train_df.dropna(subset=[label_col]).copy()
    sequences = motor_df['test_condition'].unique()
    fail_seqs = [s for s in sequences
                 if (motor_df[motor_df['test_condition'] == s][label_col] == 1).any()]
    norm_seqs = [s for s in sequences if s not in fail_seqs]

    print()
    print(f'[Split] {len(sequences)} secuencias: {len(fail_seqs)} con fallo, {len(norm_seqs)} normales')

    if len(fail_seqs) > 1:
        nv = max(1, int(0.2 * len(fail_seqs)))
        tr_f, va_f = train_test_split(fail_seqs, test_size=nv, random_state=42)
    else:
        tr_f, va_f = fail_seqs, []
    if len(norm_seqs) > 1:
        nv = max(1, int(0.2 * len(norm_seqs)))
        tr_n, va_n = train_test_split(norm_seqs, test_size=nv, random_state=42)
    else:
        tr_n, va_n = norm_seqs, []

    train_seqs = tr_f + tr_n
    val_seqs   = va_f + va_n
    print(f'  -> train: {len(train_seqs):3d} ({len(tr_f)}F + {len(tr_n)}N)')
    print(f'  -> val  : {len(val_seqs):3d} ({len(va_f)}F + {len(va_n)}N)')

    raw_train = motor_df[motor_df['test_condition'].isin(train_seqs)].copy()
    raw_val   = motor_df[motor_df['test_condition'].isin(val_seqs)].copy()

    # Inyeccion solo para motores supervisados
    if motor_id not in BAD_MOTORS:
        temp_col    = f'data_motor_{motor_id}_temperature'
        n_injected  = 0
        last_before = None
        last_after  = None
        if temp_col in raw_train.columns:
            ti = raw_train.copy()
            for seq in train_seqs:
                sm = ti['test_condition'] == seq
                if (ti.loc[sm, label_col] == 1).any():
                    tv = ti.loc[sm, temp_col].values
                    lv = ti.loc[sm, label_col].values
                    last_before = tv[lv == 1].copy()
                    ti.loc[sm, temp_col] = inject_failure(tv, lv)
                    last_after  = ti.loc[sm & (ti[label_col] == 1), temp_col].values
                    n_injected += 1
                    t  = ti.loc[sm, temp_col]
                    dc = f'data_motor_{motor_id}_temperature_diff'
                    if dc in ti.columns:
                        ti.loc[sm, dc] = t.diff(20).fillna(0).values
                    for w, suf in [(5, '5'), (20, '20')]:
                        for stat in ['mean', 'max', 'std']:
                            rc = f'data_motor_{motor_id}_temperature_roll_{stat}_{suf}'
                            if rc not in ti.columns: continue
                            if stat == 'mean':  ti.loc[sm, rc] = t.rolling(w, min_periods=1).mean().values
                            elif stat == 'max': ti.loc[sm, rc] = t.rolling(w, min_periods=1).max().values
                            else:               ti.loc[sm, rc] = t.rolling(w, min_periods=1).std().fillna(0).values
            raw_train = ti
        print()
        print(f'[Inyeccion] {n_injected} secuencias modificadas (val intacta)')
        if last_before is not None:
            print(f'  antes  -> [{last_before.min():.2f}, {last_before.max():.2f}]')
            print(f'  despues-> [{last_after.min():.2f},  {last_after.max():.2f}]')

    X_train = raw_train[features]
    y_train = raw_train[label_col]
    n0, n1  = (y_train == 0).sum(), (y_train == 1).sum()
    total   = n0 + n1
    print()
    print(f'[Balance]  0: {n0:6d} ({100*n0/total:.1f}%)   1: {n1:5d} ({100*n1/total:.1f}%)')

    X_val = raw_val[features]   if val_seqs else None
    y_val = raw_val[label_col]  if val_seqs else None

    closing_opts = [False, True]

    # ============================================================
    # ANOMALY DETECTION  (motores en BAD_MOTORS)
    # ============================================================
    if motor_id in BAD_MOTORS:
        X_normal = X_train[y_train == 0]
        print()
        print(f'[IsolationForest] entrenando sobre {len(X_normal)} muestras normales...')
        iso = IsolationForest(n_estimators=200, contamination='auto', random_state=42)
        iso.fit(X_normal)

        train_scores   = -iso.decision_function(X_train)
        pct_values     = [50, 60, 70, 75, 80, 85, 90, 92, 95, 97, 99]
        thresholds_iso = [float(np.percentile(train_scores, p)) for p in pct_values]

        best_score, best_info = -1.0, {}

        if X_val is not None:
            val_scores = -iso.decision_function(X_val)
            nv0, nv1   = (y_val == 0).sum(), (y_val == 1).sum()
            print(f'[Val] {nv0} normales, {nv1} fallos')
            if nv1 > 0:
                ns = val_scores[y_val == 0]
                fs = val_scores[y_val == 1]
                print(f'  normales -> media={ns.mean():.4f}  max={ns.max():.4f}')
                print(f'  fallos   -> media={fs.mean():.4f}  max={fs.max():.4f}')
                sep_ok = fs.mean() > ns.mean()
                print(f'  {"SEPARABLE: fallos mas anomalos" if sep_ok else "SIN SENAL: fallos no destacan"}')
            print()
            print(f'  {"Pct":>4} {"Thr":>9} {"Closing":>9} {"F1":>8}')
            print('  ' + '-' * 35)
            results_iso = []
            for p, t in zip(pct_values, thresholds_iso):
                base = (val_scores >= t).astype(int)
                for use_cl in closing_opts:
                    pred = binary_closing(base, np.ones(5)).astype(int) if use_cl else base
                    sc   = f1_score(y_val, pred, pos_label=1, zero_division=0)
                    results_iso.append((p, t, use_cl, sc))
                    if sc > best_score:
                        best_score = sc
                        best_info  = {'type': 'anomaly', 'threshold': t, 'closing': use_cl}
            for p, t, use_cl, sc in results_iso:
                thr_best = best_info.get('threshold', float('nan'))
                mark = '  <-- best' if (abs(t - thr_best) < 1e-10 and use_cl == best_info.get('closing')) else ''
                print(f'  {p:>4} {t:>9.4f} {str(use_cl):>9} {sc:>8.4f}{mark}')
        else:
            best_info = {'type': 'anomaly', 'threshold': thresholds_iso[8], 'closing': True}

        models[motor_id] = {
            'type':        'anomaly',
            'model':       iso,
            'threshold':   best_info['threshold'],
            'use_closing': best_info['closing'],
        }
        val_f1_per_motor[motor_id] = best_score if X_val is not None else None

        print()
        print(f'[Resultado] val F1 = {best_score:.4f}' if X_val is not None else '[Resultado] sin val set')
        print(f'[Config]    {best_info}')

        thr    = best_info['threshold']
        cl     = best_info['closing']
        pred_t = (train_scores >= thr).astype(int)
        if cl:
            pred_t = binary_closing(pred_t, np.ones(5)).astype(int)
        print()
        print('--- Training report ---')
        print(classification_report(y_train, pred_t, zero_division=0))

        if X_val is not None:
            pred_v = (val_scores >= thr).astype(int)
            if cl:
                pred_v = binary_closing(pred_v, np.ones(5)).astype(int)
            print('--- Validation report ---')
            print(classification_report(y_val, pred_v, zero_division=0))

    # ============================================================
    # SUPERVISED XGBOOST  (motores 1, 2, 5)
    # ============================================================
    else:
        ratio = min(n0 / n1, 20.0) if n1 > 0 else 1.0
        sw    = np.where(y_train == 1, ratio, 1.0)
        print(f'[Pesos] {ratio:.1f}x en clase 1  (cap 20x)')

        thresholds = [0.005, 0.01, 0.02, 0.03, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

        if motor_id == 1:
            param_combos = [{'C': c} for c in [0.01, 0.1, 1.0, 10.0]]
            model_name   = 'LogisticRegression'
        else:
            pg   = {'learning_rate': [0.02, 0.05], 'n_estimators': [100, 150],
                    'max_depth': [3, 4, 5], 'min_child_weight': [1, 10]}
            ks, vs = zip(*pg.items())
            param_combos = [dict(zip(ks, v)) for v in itertools.product(*vs)]
            model_name   = 'XGBoost'

        n_cfg = len(param_combos) * len(thresholds) * len(closing_opts)
        print()
        print(f'[Grid search] {model_name} -- {len(param_combos)} params x {len(thresholds)} thresholds x {len(closing_opts)} closing = {n_cfg} configs')
        print()

        best_model, best_score, best_info = None, -1.0, {}

        for i, params in enumerate(param_combos):
            if motor_id == 1:
                mdl = Pipeline([('sc', StandardScaler()),
                                ('lr', LogisticRegression(C=params['C'], random_state=42, max_iter=1000))])
                mdl.fit(X_train, y_train, lr__sample_weight=sw)
            else:
                mdl = xgb.XGBClassifier(random_state=42, eval_metric='logloss',
                                         verbosity=0, tree_method='hist', **params)
                mdl.fit(X_train, y_train, sample_weight=sw)

            if X_val is not None:
                probs = mdl.predict_proba(X_val)[:, 1]
                for t in thresholds:
                    base = (probs >= t).astype(int)
                    for use_cl in closing_opts:
                        pred = binary_closing(base, np.ones(5)).astype(int) if use_cl else base
                        sc   = f1_score(y_val, pred, pos_label=1, zero_division=0)
                        if sc > best_score:
                            best_score = sc
                            best_model = mdl
                            best_info  = {'type': 'supervised', 'params': params, 'threshold': t, 'closing': use_cl}
                print(f'  combo {i+1:2d}/{len(param_combos)}  mejor F1: {best_score:.4f}  {params}')
            else:
                best_model = mdl
                best_info  = {'type': 'supervised', 'params': params, 'threshold': 0.5, 'closing': True}
                print(f'  combo {i+1:2d}/{len(param_combos)}  (sin val)')
                break

        models[motor_id] = {
            'type':        'supervised',
            'model':       best_model,
            'threshold':   best_info.get('threshold', 0.5),
            'use_closing': best_info.get('closing', True),
        }
        val_f1_per_motor[motor_id] = best_score if X_val is not None else None

        print()
        print(f'[Resultado] val F1 = {best_score:.4f}' if X_val is not None else '[Resultado] sin val set')
        print(f'[Config]    {best_info}')

        thr    = best_info.get('threshold', 0.5)
        cl     = best_info.get('closing', True)
        pt     = best_model.predict_proba(X_train)[:, 1]
        pred_t = (pt >= thr).astype(int)
        if cl:
            pred_t = binary_closing(pred_t, np.ones(5)).astype(int)
        print()
        print('--- Training report ---')
        print(classification_report(y_train, pred_t, zero_division=0))

        if X_val is not None:
            pv     = best_model.predict_proba(X_val)[:, 1]
            pred_v = (pv >= thr).astype(int)
            if cl:
                pred_v = binary_closing(pred_v, np.ones(5)).astype(int)
            print('--- Validation report ---')
            print(classification_report(y_val, pred_v, zero_division=0))


print('train_motor() definida')

train_motor() definida


## 6. Entrenamiento por motor
Cada celda es independiente. Puedes re-ejecutar un motor solo sin repetir los demas.

In [45]:
train_motor(1)


  MOTOR 1  (supervisado)

[Split] 23 secuencias: 2 con fallo, 21 normales
  -> train:  18 (1F + 17N)
  -> val  :   5 (1F + 4N)

[Inyeccion] 1 secuencias modificadas (val intacta)
  antes  -> [2.00, 18.00]
  despues-> [2.00,  20.00]

[Balance]  0:  28678 (95.7%)   1:  1292 (4.3%)
[Pesos] 20.0x en clase 1  (cap 20x)

[Grid search] LogisticRegression -- 4 params x 14 thresholds x 2 closing = 112 configs

  combo  1/4  mejor F1: 0.5679  {'C': 0.01}
  combo  2/4  mejor F1: 0.5679  {'C': 0.1}
  combo  3/4  mejor F1: 0.5679  {'C': 1.0}
  combo  4/4  mejor F1: 0.5679  {'C': 10.0}

[Resultado] val F1 = 0.5679
[Config]    {'type': 'supervised', 'params': {'C': 0.01}, 'threshold': 0.8, 'closing': False}

--- Training report ---
              precision    recall  f1-score   support

           0       1.00      0.99      0.99     28678
           1       0.78      0.98      0.87      1292

    accuracy                           0.99     29970
   macro avg       0.89      0.99      0.93     29970


In [46]:
train_motor(2)


  MOTOR 2  (supervisado)

[Split] 23 secuencias: 2 con fallo, 21 normales
  -> train:  18 (1F + 17N)
  -> val  :   5 (1F + 4N)

[Inyeccion] 1 secuencias modificadas (val intacta)
  antes  -> [-4.00, 3.00]
  despues-> [-2.00,  8.00]

[Balance]  0:  23318 (77.8%)   1:  6652 (22.2%)
[Pesos] 3.5x en clase 1  (cap 20x)

[Grid search] XGBoost -- 24 params x 14 thresholds x 2 closing = 672 configs

  combo  1/24  mejor F1: 0.0170  {'learning_rate': 0.02, 'n_estimators': 100, 'max_depth': 3, 'min_child_weight': 1}
  combo  2/24  mejor F1: 0.0170  {'learning_rate': 0.02, 'n_estimators': 100, 'max_depth': 3, 'min_child_weight': 10}
  combo  3/24  mejor F1: 0.0170  {'learning_rate': 0.02, 'n_estimators': 100, 'max_depth': 4, 'min_child_weight': 1}
  combo  4/24  mejor F1: 0.0170  {'learning_rate': 0.02, 'n_estimators': 100, 'max_depth': 4, 'min_child_weight': 10}
  combo  5/24  mejor F1: 0.0170  {'learning_rate': 0.02, 'n_estimators': 100, 'max_depth': 5, 'min_child_weight': 1}
  combo  6/24  me

In [47]:
train_motor(3)


  MOTOR 3  (anomalia)

[Split] 23 secuencias: 2 con fallo, 21 normales
  -> train:  18 (1F + 17N)
  -> val  :   5 (1F + 4N)

[Balance]  0:  29926 (99.9%)   1:    44 (0.1%)

[IsolationForest] entrenando sobre 29926 muestras normales...
[Val] 9256 normales, 83 fallos
  normales -> media=-0.1456  max=0.1248
  fallos   -> media=0.0026  max=0.1356
  SEPARABLE: fallos mas anomalos

   Pct       Thr   Closing       F1
  -----------------------------------
    50   -0.1296     False   0.0705
    50   -0.1296      True   0.0697
    60   -0.0832     False   0.2744
    60   -0.0832      True   0.2690
    70   -0.0229     False   0.2640
    70   -0.0229      True   0.2973
    75    0.0079     False   0.2322
    75    0.0079      True   0.2374
    80    0.0340     False   0.2870
    80    0.0340      True   0.2907
    85    0.0425     False   0.3229
    85    0.0425      True   0.3128
    90    0.0586     False   0.3067
    90    0.0586      True   0.3390  <-- best
    92    0.0718     False   0.2

In [48]:
train_motor(4)


  MOTOR 4  (anomalia)

[Split] 23 secuencias: 2 con fallo, 21 normales
  -> train:  18 (1F + 17N)
  -> val  :   5 (1F + 4N)

[Balance]  0:  23318 (77.8%)   1:  6652 (22.2%)

[IsolationForest] entrenando sobre 23318 muestras normales...
[Val] 9252 normales, 87 fallos
  normales -> media=-0.0797  max=0.1983
  fallos   -> media=0.1018  max=0.1527
  SEPARABLE: fallos mas anomalos

   Pct       Thr   Closing       F1
  -----------------------------------
    50   -0.0440     False   0.0632
    50   -0.0440      True   0.0628
    60   -0.0255     False   0.0643
    60   -0.0255      True   0.0643
    70    0.0042     False   0.0732
    70    0.0042      True   0.0720
    75    0.0156     False   0.0749
    75    0.0156      True   0.0740
    80    0.0308     False   0.0833
    80    0.0308      True   0.0802
    85    0.0457     False   0.0917
    85    0.0457      True   0.0894
    90    0.0629     False   0.0991
    90    0.0629      True   0.0993
    92    0.0728     False   0.1027
    9

In [36]:
train_motor(5)


  MOTOR 5  (supervisado)

[Split] 23 secuencias: 2 con fallo, 21 normales
  -> train:  18 (1F + 17N)
  -> val  :   5 (1F + 4N)

[Inyeccion] 1 secuencias modificadas (val intacta)
  antes  -> [-1.00, 1.00]
  despues-> [-1.00,  1.00]

[Balance]  0:  29900 (99.8%)   1:    70 (0.2%)
[Pesos] 20.0x en clase 1  (cap 20x)

[Grid search] XGBoost -- 24 params x 14 thresholds x 2 closing = 672 configs

  combo  1/24  mejor F1: 0.4298  {'learning_rate': 0.02, 'n_estimators': 100, 'max_depth': 3, 'min_child_weight': 1}
  combo  2/24  mejor F1: 0.4298  {'learning_rate': 0.02, 'n_estimators': 100, 'max_depth': 3, 'min_child_weight': 10}
  combo  3/24  mejor F1: 0.4298  {'learning_rate': 0.02, 'n_estimators': 100, 'max_depth': 4, 'min_child_weight': 1}
  combo  4/24  mejor F1: 0.4298  {'learning_rate': 0.02, 'n_estimators': 100, 'max_depth': 4, 'min_child_weight': 10}
  combo  5/24  mejor F1: 0.4298  {'learning_rate': 0.02, 'n_estimators': 100, 'max_depth': 5, 'min_child_weight': 1}
  combo  6/24  me

In [37]:
train_motor(6)


  MOTOR 6  (anomalia)

[Split] 23 secuencias: 7 con fallo, 16 normales
  -> train:  19 (6F + 13N)
  -> val  :   4 (1F + 3N)

[Balance]  0:  24193 (94.9%)   1:  1313 (5.1%)

[IsolationForest] entrenando sobre 24193 muestras normales...
[Val] 13184 normales, 619 fallos
  normales -> media=-0.0770  max=0.2085
  fallos   -> media=-0.0259  max=0.1586
  SEPARABLE: fallos mas anomalos

   Pct       Thr   Closing       F1
  -----------------------------------
    50   -0.1024     False   0.1448
    50   -0.1024      True   0.1430
    60   -0.0821     False   0.1716
    60   -0.0821      True   0.1688
    70   -0.0656     False   0.1711
    70   -0.0656      True   0.1969  <-- best
    75   -0.0468     False   0.1552
    75   -0.0468      True   0.1497
    80   -0.0023     False   0.0754
    80   -0.0023      True   0.0825
    85    0.0439     False   0.0793
    85    0.0439      True   0.0748
    90    0.0840     False   0.0712
    90    0.0840      True   0.0764
    92    0.1010     False   

## 7. Resumen global de F1

In [38]:
print('========== VALIDATION F1 SUMMARY ==========')
scored = {}
for mid in range(1, 7):
    s = val_f1_per_motor.get(mid)
    if s is not None:
        scored[mid] = s
        print(f'  Motor {mid}: {s:.4f}')
    else:
        print(f'  Motor {mid}: sin val set')
if scored:
    gf1 = sum(scored.values()) / len(scored)
    print('  ---')
    print(f'  Global mean F1 ({len(scored)} motores): {gf1:.4f}')
print('===========================================')

========== VALIDATION F1 SUMMARY ==========
  Motor 1: 0.5750
  Motor 2: 0.4822
  Motor 3: 0.3390
  Motor 4: 0.2656
  Motor 5: 0.6265
  Motor 6: 0.1969
  ---
  Global mean F1 (6 motores): 0.4142


## 8. Generar submission
Pon `EXCLUDE_MOTOR = N` (1-6) para el truco Kaggle (motor a -1, no gasta submission).
Pon `EXCLUDE_MOTOR = None` para el CSV final.

In [39]:
EXCLUDE_MOTOR = None  # 1-6 para truco Kaggle, None para submission final

sample_sub = pd.read_csv(base_dir + '/sample_submission.csv')
final_sub  = sample_sub.copy()

for test_id in sample_sub['test_condition'].unique():
    sub_mask = sample_sub['test_condition'] == test_id
    mtd      = test_df[test_df['test_condition'] == test_id].sort_values('time')
    exp_len  = sub_mask.sum()

    if len(mtd) == 0:
        for mid in range(1, 7):
            final_sub.loc[sub_mask, f'data_motor_{mid}_label'] = 0
        continue

    for mid in range(1, 7):
        feats = [f'data_motor_{mid}_{f}' for f in MOTOR_FEATURES[mid]]
        if mid not in models or not all(f in mtd.columns for f in feats):
            final_sub.loc[sub_mask, f'data_motor_{mid}_label'] = 0
            continue

        info = models[mid]
        X    = mtd[feats]

        if info.get('type', 'supervised') == 'anomaly':
            scores = -info['model'].decision_function(X)
            preds  = (scores >= info['threshold']).astype(int)
        else:
            probs = info['model'].predict_proba(X)[:, 1]
            preds = (probs >= info['threshold']).astype(int)

        if info['use_closing']:
            preds = binary_closing(preds, structure=np.ones(5)).astype(int)

        if len(preds) != exp_len:
            print(f'Length mismatch {test_id} motor {mid}: {len(preds)} vs {exp_len}')
            preds = preds[:exp_len] if len(preds) > exp_len else np.pad(preds, (0, exp_len - len(preds)), 'constant')

        final_sub.loc[sub_mask, f'data_motor_{mid}_label'] = preds

for mid in range(1, 7):
    final_sub[f'data_motor_{mid}_label'] = final_sub[f'data_motor_{mid}_label'].astype(int)

if EXCLUDE_MOTOR is not None:
    print(f'Truco Kaggle: motor {EXCLUDE_MOTOR} -> -1')
    final_sub[f'data_motor_{EXCLUDE_MOTOR}_label'] = -1
    out = f'./motor_excluded_{EXCLUDE_MOTOR}_submission.csv'
else:
    out = './motor_submission.csv'

final_sub.to_csv(out, index=False)
print(f'Guardado: {out}')
print(f'Shape: {final_sub.shape}')
print()
print('Primeras filas:')
final_sub.head()

Guardado: ./motor_submission.csv
Shape: (14157, 8)

Primeras filas:


,idx,data_motor_1_label,data_motor_2_label,data_motor_3_label,data_motor_4_label,data_motor_5_label,data_motor_6_label,test_condition
0,0,0,0,0,0,0,0,20240527_094865
1,1,0,0,0,0,0,0,20240527_094865
2,2,0,0,0,0,0,0,20240527_094865
3,3,0,0,0,0,0,0,20240527_094865
4,4,0,0,0,0,0,0,20240527_094865
